# Task 4: Vectorising MNIST and metric learning

Part A turns images into vectors and uses vector similarity to classify digits.
Part B trains a small network so that the vectors themselves become better (metric learning).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from sklearn.decomposition import PCA

train_set = datasets.MNIST(root="/tmp/", train=True, download=True)
test_set = datasets.MNIST(root="/tmp/", train=False, download=True)

N_TRAIN, N_TEST = 5000, 500
train_images = train_set.data[:N_TRAIN].numpy()   # (N, 28, 28) uint8
train_labels = train_set.targets[:N_TRAIN].numpy()
test_images = test_set.data[:N_TEST].numpy()
test_labels = test_set.targets[:N_TEST].numpy()

## Part A: images as vectors

In [ ]:
print(f"Train images: {train_images.shape}, test images: {test_images.shape}")

In [ ]:
# 1. Turn each 28x28 image into a flat vector of 784 floats in [0, 1]
def vectorize(images):
    return ...

train_vecs = ...
test_vecs = ...
print(f"Vectorized shape: {train_vecs.shape}")

In [ ]:
# 2. Average vector of each digit, shape (10, 784)
def class_means(vecs, labels):
    return np.stack([... for d in range(10)])

means = ...

# Show each mean vector reshaped back to an image
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for d, ax in enumerate(axes):
    ax.imshow(..., cmap="gray")
    ax.set_title(str(d))
    ax.axis("off")
plt.tight_layout()
plt.savefig("digit_means.png")
plt.show()

In [ ]:
# 3. Reduce the 784-dim vectors to 2D with PCA and plot them
pca = ...
reduced = ...

plt.figure(figsize=(8, 6))
scatter = plt.scatter(..., ..., c=..., cmap="tab10", s=5)
plt.colorbar(scatter, label="Digit")
plt.title("PCA of vectorized MNIST")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.tight_layout()
plt.savefig("mnist_pca.png")
plt.show()

In [ ]:
# 4. Cosine similarity between EVERY row of a and EVERY row of b, with no Python loops
#    a: (n, d), b: (m, d) -> result: (n, m)
def cosine_similarity_matrix(a, b):
    # hint: normalise rows (np.linalg.norm with axis=1, keepdims=True), then one matrix multiplication
    ...

In [ ]:
# 5. "Training" = storing vectors. Predict each test vector's label from its most similar train vector
def predict_nearest(test_vecs, train_vecs, train_labels):
    sims = ...
    return ...

In [ ]:
# 6. Same idea, but compare against the 10 class mean vectors only
def predict_by_means(test_vecs, means):
    sims = ...
    return ...

In [ ]:
nn_preds = ...
mean_preds = ...

nn_accuracy = ...
mean_accuracy = ...
print(f"Nearest neighbour accuracy: {nn_accuracy:.4f}")
print(f"Class-mean accuracy: {mean_accuracy:.4f}")

## Part B: metric learning

Raw pixels are a fixed vector space: two 7s written in different places can be far apart.
In metric learning we train a network `f` that maps each image to a new vector (an *embedding*),
so that images of the same digit end up close and different digits end up far apart.

We use the **triplet loss**: for an *anchor* image, a *positive* (same digit) and a *negative* (different digit),

$$L = \max(0,\ d(f(a), f(p)) - d(f(a), f(n)) + \text{margin})$$

The loss is zero once the negative is at least `margin` further away than the positive.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Embedding network: 784 -> 64, output normalised to unit length so cosine similarity = dot product
class EmbeddingNet(nn.Module):
    def __init__(self, in_dim=784, emb_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            # Linear(in_dim -> 256), ReLU, Linear(256 -> emb_dim)
            ...
        )

    def forward(self, x):
        # hint: nn.functional.normalize(..., dim=1)
        return ...

embed_net = EmbeddingNet().to(device)

In [ ]:
# For each anchor index pick a random positive (same label) and negative (different label)
def sample_triplets(labels, batch_size):
    anchors = rng.integers(0, len(labels), batch_size)
    # hint: rng.choice(np.flatnonzero(...)) picks a random index where the condition is True
    positives = np.array([... for a in anchors])
    negatives = np.array([... for a in anchors])
    return anchors, positives, negatives

In [ ]:
loss_fn = nn.TripletMarginLoss(margin=0.5)
optimizer = torch.optim.Adam(embed_net.parameters(), lr=1e-3)
X_train = torch.tensor(train_vecs, device=device)

losses = []
for step in range(1000):
    a, p, n = sample_triplets(train_labels, batch_size=128)
    loss = loss_fn(..., ..., ...)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 200 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Triplet loss")
plt.title("Metric learning training")
plt.show()

In [ ]:
# Embed train and test images with the trained network
def embed(vecs):
    with torch.no_grad():
        return embed_net(torch.tensor(vecs, device=device)).cpu().numpy()

train_emb = embed(train_vecs)
test_emb = embed(test_vecs)
print(f"Embedding shape: {train_emb.shape}")

In [ ]:
# Same nearest-neighbour classifier as in Part A, but on learned embeddings
emb_preds = ...
emb_accuracy = ...
print(f"Raw pixels (784 dims) nearest neighbour accuracy: {nn_accuracy:.4f}")
print(f"Learned embedding (64 dims) nearest neighbour accuracy: {emb_accuracy:.4f}")

In [ ]:
# Compare PCA of raw pixels with PCA of learned embeddings
emb_reduced = PCA(n_components=2).fit_transform(train_emb)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title in [(axes[0], reduced, "Raw pixels"), (axes[1], emb_reduced, "Learned embedding")]:
    sc = ax.scatter(data[:, 0], data[:, 1], c=train_labels, cmap="tab10", s=5)
    ax.set_title(f"PCA: {title}")
fig.colorbar(sc, ax=axes, label="Digit")
plt.savefig("metric_learning_pca.png")
plt.show()

**Questions**
1. Why do digit clusters separate better in the learned embedding, even though it has fewer dimensions?
2. What happens to accuracy if you set `margin` to 0? To 2?
3. The negatives are random. Why could picking *hard* negatives (the closest wrong digit) train faster?